# Example 9.6 / 1: Read Values into Parameters

Companion notebook for **9.6 The `read` Command and File Conversion** in the AMPL lab Documentation, section "Read Values into Parameters". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Read Values into Parameters

One `read` assigns all four values in argument order, the rates in the order of the ordered set
`PROD`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD ordered;                 # products, in the file's order
param avail >= 0;                 # hours available
param rate {PROD} > 0;            # tons per hour

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("steel.txt", "w") as f:
    f.write("40\n200 140 160\n")
ampl.set["PROD"] = ["bands", "coils", "plate"]

In [5]:
# run.py
ampl.eval("read avail, {p in PROD} rate[p] < steel.txt;")
print(ampl.param["avail"].value())
print(ampl.param["rate"].to_dict())

40
{'bands': 200, 'coils': 140, 'plate': 160}


### Pure AMPL: data.dat and model.run

amplpy has no method for `read`, so it goes through `ampl.eval`. Line breaks in the file do
not matter; only the order of the values does.

Pure AMPL, with the same `model.mod` and `steel.txt`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
read avail, {p in PROD} rate[p] < steel.txt;
display avail, rate;

avail = 40

rate [*] :=
bands  200
coils  140
plate  160
;

